In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../aruba_cleaned.csv")
df

,date,time,sensor,value,timestamp,sensor_type,hour,time_period,activity_label,activity_state
0,2010-11-04,00:03:50.209589,M003,ON Sleeping begin,2010-11-04 00:03:50.209589,M,0,night,Sleeping,begin
1,2010-11-04,00:03:57.399391,M003,OFF,2010-11-04 00:03:57.399391,M,0,night,NaN,NaN
2,2010-11-04,02:32:33.351906,M003,ON,2010-11-04 02:32:33.351906,M,2,night,NaN,NaN
3,2010-11-04,02:32:38.895958,M003,OFF,2010-11-04 02:32:38.895958,M,2,night,NaN,NaN
4,2010-11-04,03:42:21.82365,M003,ON,2010-11-04 03:42:21.823650,M,3,night,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
1602976,2011-06-11,22:33:01.209214,M002,ON,2011-06-11 22:33:01.209214,M,22,night,NaN,NaN
1602977,2011-06-11,22:33:02.952009,M003,OFF,2011-06-11 22:33:02.952009,M,22,night,NaN,NaN
1602978,2011-06-11,22:33:03.332876,M002,OFF,2011-06-11 22:33:03.332876,M,22,night,NaN,NaN
1602979,2011-06-11,23:20:31.432636,M003,ON,2011-06-11 23:20:31.432636,M,23,night,NaN,NaN


In [4]:
activities = df["activity_label"].dropna().unique()

print(activities)

['Sleeping' 'Bed_to_Toilet' 'Meal_Preparation' 'Relax' 'Housekeeping'
 'Eating' 'Wash_Dishes' 'OPEN Leave_Home' 'CLOSE Leave_Home'
 'OPEN Enter_Home' 'CLOSE Enter_Home' 'Work' 'Respirate' 'Leave_Home'
 'Enter_Home' 'O Relax' 'OF Meal_Preparation']


In [6]:
# Make sure timestamp is datetime
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Keep only rows that have an activity
activity_df = df[df["activity_label"].notna()].copy()

# Sort by time
activity_df = activity_df.sort_values("timestamp")

activities = []

for activity, group in activity_df.groupby("activity_label"):
    
    group = group.sort_values("timestamp")
    start_time = None

    for _, row in group.iterrows():

        if row["activity_state"] == "begin":
            start_time = row["timestamp"]

        elif row["activity_state"] == "end" and start_time is not None:

            end_time = row["timestamp"]

            duration = end_time - start_time

            activities.append({
                "activity": activity,
                "start": start_time,
                "end": end_time,
                "duration": duration
            })

            start_time = None

# Create DataFrame
activity_duration_df = pd.DataFrame(activities)

# Convert duration to minutes
activity_duration_df["duration_minutes"] = (
    activity_duration_df["duration"].dt.total_seconds() / 60
)

print(activity_duration_df.head())

        activity                      start                        end  \
0  Bed_to_Toilet 2010-11-04 05:40:51.303739 2010-11-04 05:43:30.279021   
1  Bed_to_Toilet 2010-11-05 04:18:09.406239 2010-11-05 04:20:50.109684   
2  Bed_to_Toilet 2010-11-06 04:23:50.544193 2010-11-06 04:27:00.048965   
3  Bed_to_Toilet 2010-11-07 00:34:48.871659 2010-11-07 00:35:36.223000   
4  Bed_to_Toilet 2010-11-07 03:43:19.802838 2010-11-07 03:46:23.886632   

                duration  duration_minutes  
0 0 days 00:02:38.975282          2.649588  
1 0 days 00:02:40.703445          2.678391  
2 0 days 00:03:09.504772          3.158413  
3 0 days 00:00:47.351341          0.789189  
4 0 days 00:03:04.083794          3.068063  


In [7]:
duration_summary = (
    activity_duration_df
    .groupby("activity")["duration_minutes"]
    .agg(
        average_duration="mean",
        min_duration="min",
        max_duration="max"
    )
    .reset_index()
)

print(duration_summary)

            activity  average_duration  min_duration  max_duration
0      Bed_to_Toilet          2.729481      0.357222      8.702630
1   CLOSE Enter_Home        829.721475    829.721475    829.721475
2   CLOSE Leave_Home        431.749579    431.749579    431.749579
3             Eating         10.149268      0.264641     69.475333
4         Enter_Home        391.872700    391.872700    391.872700
5       Housekeeping         20.324018      1.536077     90.010639
6         Leave_Home       7846.165769      0.014001  15692.317537
7   Meal_Preparation          7.838226      0.126168    326.200545
8              Relax         33.431752      0.944290    255.024874
9          Respirate          8.564308      1.638024     24.919531
10          Sleeping        241.214425      0.555374    646.396949
11       Wash_Dishes          7.246125      0.554909     36.654173
12              Work         17.080463      0.913070    116.940938


In [8]:
activity_counts = df["activity_label"].value_counts()

print(activity_counts)

activity_label
Relax                  5837
Meal_Preparation       3211
Sleeping                801
Eating                  514
CLOSE Enter_Home        431
OPEN Leave_Home         427
CLOSE Leave_Home        426
OPEN Enter_Home         425
Work                    342
Bed_to_Toilet           314
Wash_Dishes             130
Housekeeping             66
Respirate                12
Leave_Home                9
Enter_Home                6
O Relax                   1
OF Meal_Preparation       1
Name: count, dtype: int64


In [9]:
activities_to_remove = [
    "Respirate",
    "Leave_Home",
    "Enter_Home",
    "O Relax",
    "OF Meal_Preparation"
]

df = df[~df["activity_label"].isin(activities_to_remove)].copy()

In [10]:
activity_counts = df["activity_label"].value_counts()

print(activity_counts)

activity_label
Relax               5837
Meal_Preparation    3211
Sleeping             801
Eating               514
CLOSE Enter_Home     431
OPEN Leave_Home      427
CLOSE Leave_Home     426
OPEN Enter_Home      425
Work                 342
Bed_to_Toilet        314
Wash_Dishes          130
Housekeeping          66
Name: count, dtype: int64


In [11]:
duration_summary = (
    activity_duration_df
    .groupby("activity")["duration_minutes"]
    .agg(
        average_duration="mean",
        min_duration="min",
        max_duration="max"
    )
    .reset_index()
)

print(duration_summary)

            activity  average_duration  min_duration  max_duration
0      Bed_to_Toilet          2.729481      0.357222      8.702630
1   CLOSE Enter_Home        829.721475    829.721475    829.721475
2   CLOSE Leave_Home        431.749579    431.749579    431.749579
3             Eating         10.149268      0.264641     69.475333
4         Enter_Home        391.872700    391.872700    391.872700
5       Housekeeping         20.324018      1.536077     90.010639
6         Leave_Home       7846.165769      0.014001  15692.317537
7   Meal_Preparation          7.838226      0.126168    326.200545
8              Relax         33.431752      0.944290    255.024874
9          Respirate          8.564308      1.638024     24.919531
10          Sleeping        241.214425      0.555374    646.396949
11       Wash_Dishes          7.246125      0.554909     36.654173
12              Work         17.080463      0.913070    116.940938


In [14]:
activity_duration_df["date"] = activity_duration_df["start"].dt.date

activity_duration_df["hour"] = activity_duration_df["start"].dt.hour

activity_duration_df["minute"] = activity_duration_df["start"].dt.minute

activity_duration_df["day_of_week"] = (
    activity_duration_df["start"].dt.day_name()
)

activity_duration_df["start_time"] = (
    activity_duration_df["start"].dt.time
)

print(activity_duration_df.head(50))

         activity                      start                        end  \
0   Bed_to_Toilet 2010-11-04 05:40:51.303739 2010-11-04 05:43:30.279021   
1   Bed_to_Toilet 2010-11-05 04:18:09.406239 2010-11-05 04:20:50.109684   
2   Bed_to_Toilet 2010-11-06 04:23:50.544193 2010-11-06 04:27:00.048965   
3   Bed_to_Toilet 2010-11-07 00:34:48.871659 2010-11-07 00:35:36.223000   
4   Bed_to_Toilet 2010-11-07 03:43:19.802838 2010-11-07 03:46:23.886632   
5   Bed_to_Toilet 2010-11-07 03:49:20.647413 2010-11-07 03:51:17.964123   
6   Bed_to_Toilet 2010-11-08 05:40:10.556874 2010-11-08 05:43:15.104099   
7   Bed_to_Toilet 2010-11-09 00:30:32.314798 2010-11-09 00:32:54.868485   
8   Bed_to_Toilet 2010-11-09 05:31:30.888221 2010-11-09 05:34:08.785188   
9   Bed_to_Toilet 2010-11-11 05:13:58.821109 2010-11-11 05:16:28.261014   
10  Bed_to_Toilet 2010-11-12 04:31:41.818859 2010-11-12 04:35:07.429031   
11  Bed_to_Toilet 2010-11-13 05:17:05.045679 2010-11-13 05:20:52.478511   
12  Bed_to_Toilet 2010-11

In [ ]:
routine_summary = (
    activity_duration_df
    .groupby("activity")
    .agg(
        average_start_hour=("hour", "mean"),
        average_duration=("duration_minutes", "mean"),
        min_duration=("duration_minutes", "min"),
        max_duration=("duration_minutes", "max"),
        frequency=("activity", "count") #el frequency of each activity yaane kam event et3ml w kheles completed
    )
    .reset_index()
)

print(routine_summary)

            activity  average_start_hour  average_duration  min_duration  \
0      Bed_to_Toilet            4.192308          2.729481      0.357222   
1   CLOSE Enter_Home           20.000000        829.721475    829.721475   
2   CLOSE Leave_Home            8.000000        431.749579    431.749579   
3             Eating           12.670588         10.149268      0.264641   
4         Enter_Home           14.000000        391.872700    391.872700   
5       Housekeeping           12.575758         20.324018      1.536077   
6         Leave_Home           13.500000       7846.165769      0.014001   
7   Meal_Preparation           12.578056          7.838226      0.126168   
8              Relax           15.271851         33.431752      0.944290   
9          Respirate           11.166667          8.564308      1.638024   
10          Sleeping            8.801008        241.214425      0.555374   
11       Wash_Dishes           14.937500          7.246125      0.554909   
12          

In [18]:
routine_data = activity_duration_df[
    [
        "date",
        "activity",
        "start",
        "duration_minutes",
        "day_of_week"
    ]
].copy()

print(routine_data.head())

         date       activity                      start  duration_minutes  \
0  2010-11-04  Bed_to_Toilet 2010-11-04 05:40:51.303739          2.649588   
1  2010-11-05  Bed_to_Toilet 2010-11-05 04:18:09.406239          2.678391   
2  2010-11-06  Bed_to_Toilet 2010-11-06 04:23:50.544193          3.158413   
3  2010-11-07  Bed_to_Toilet 2010-11-07 00:34:48.871659          0.789189   
4  2010-11-07  Bed_to_Toilet 2010-11-07 03:43:19.802838          3.068063   

  day_of_week  
0    Thursday  
1      Friday  
2    Saturday  
3      Sunday  
4      Sunday  


In [27]:
daily_frequency = (
    routine_data
    .groupby(["date", "activity"])
    .size()
    .reset_index(name="count")
)

print(daily_frequency.tail())

            date          activity  count
1088  2011-06-10              Work      1
1089  2011-06-11            Eating      1
1090  2011-06-11  Meal_Preparation      8
1091  2011-06-11             Relax     17
1092  2011-06-11              Work      1


In [28]:
average_frequency = (
    daily_frequency
    .groupby("activity")["count"]
    .mean()
    .reset_index(name="average_daily_frequency")
)

print(average_frequency)

            activity  average_daily_frequency
0      Bed_to_Toilet                 1.172932
1   CLOSE Enter_Home                 1.000000
2   CLOSE Leave_Home                 1.000000
3             Eating                 1.861314
4         Enter_Home                 1.000000
5       Housekeeping                 1.178571
6         Leave_Home                 1.000000
7   Meal_Preparation                 7.283105
8              Relax                13.269406
9          Respirate                 1.000000
10          Sleeping                 1.965347
11       Wash_Dishes                 1.254902
12              Work                 1.838710


In [29]:
activities_to_remove = [
    "CLOSE Enter_Home",
    "CLOSE Leave_Home",
    "OPEN Enter_Home",
    "OPEN Leave_Home",
    "Enter_Home",
    "Leave_Home",
    "Respirate"
] #shelna el door activities for now

df = df[~df["activity_label"].isin(activities_to_remove)].copy()

print(df["activity_label"].value_counts())

activity_label
Relax               5837
Meal_Preparation    3211
Sleeping             801
Eating               514
Work                 342
Bed_to_Toilet        314
Wash_Dishes          130
Housekeeping          66
Name: count, dtype: int64


In [31]:
# Activities we don't want in the routine baseline
activities_to_remove = [
    "CLOSE Enter_Home",
    "CLOSE Leave_Home",
    "OPEN Enter_Home",
    "OPEN Leave_Home",
    "Enter_Home",
    "Leave_Home",
    "Respirate"
]

# Remove them from df
df = df[~df["activity_label"].isin(activities_to_remove)].copy()


# =========================================
# Recreate activity_duration_df
# =========================================

df["timestamp"] = pd.to_datetime(df["timestamp"])

activity_df = df[df["activity_label"].notna()].copy()
activity_df = activity_df.sort_values("timestamp")

activities = []

for activity, group in activity_df.groupby("activity_label"):

    group = group.sort_values("timestamp")
    start_time = None

    for _, row in group.iterrows():

        if row["activity_state"] == "begin":
            start_time = row["timestamp"]

        elif row["activity_state"] == "end" and start_time is not None:

            end_time = row["timestamp"]

            activities.append({
                "activity": activity,
                "start": start_time,
                "end": end_time,
                "duration": end_time - start_time
            })

            start_time = None


activity_duration_df = pd.DataFrame(activities)

# Duration in minutes
activity_duration_df["duration_minutes"] = (
    activity_duration_df["duration"].dt.total_seconds() / 60
)

# Add time features
activity_duration_df["date"] = activity_duration_df["start"].dt.date
activity_duration_df["hour"] = activity_duration_df["start"].dt.hour
activity_duration_df["minute"] = activity_duration_df["start"].dt.minute

In [ ]:
#baseline table
daily_frequency = (
    activity_duration_df
    .groupby(["date", "activity"])
    .size()
    .reset_index(name="daily_frequency")
)

average_frequency = (
    daily_frequency
    .groupby("activity")["daily_frequency"]
    .mean()
    .reset_index(name="average_daily_frequency")
)

duration_stats = (
    activity_duration_df
    .groupby("activity")
    .agg(
        average_duration=("duration_minutes", "mean"),
        min_duration=("duration_minutes", "min"),
        max_duration=("duration_minutes", "max"),
        average_start_hour=("hour", "mean")
    )
    .reset_index()
)

baseline_table = average_frequency.merge(
    duration_stats,
    on="activity"
)

baseline_table = baseline_table.round(2)

print(baseline_table)

           activity  average_daily_frequency  average_duration  min_duration  \
0     Bed_to_Toilet                     1.17              2.73          0.36   
1            Eating                     1.86             10.15          0.26   
2      Housekeeping                     1.18             20.32          1.54   
3  Meal_Preparation                     7.28              7.84          0.13   
4             Relax                    13.27             33.43          0.94   
5          Sleeping                     1.97            241.21          0.56   
6       Wash_Dishes                     1.25              7.25          0.55   
7              Work                     1.84             17.08          0.91   

   max_duration  average_start_hour  
0          8.70                4.19  
1         69.48               12.67  
2         90.01               12.58  
3        326.20               12.58  
4        255.02               15.27  
5        646.40                8.80  
6         36.

In [ ]:
# =========================================
# Daily Routine Table
# =========================================
#What is normal for this person overall


daily_routine = (
    activity_duration_df
    .groupby(["date", "activity"])
    .agg(
        frequency=("activity", "count"),
        total_duration=("duration_minutes", "sum"),
        average_duration=("duration_minutes", "mean"),
        average_start_hour=("hour", "mean")
    )
    .reset_index()
)

# Round the numbers
daily_routine = daily_routine.round(2)

print(daily_routine.head(20))

          date          activity  frequency  total_duration  average_duration  \
0   2010-11-04     Bed_to_Toilet          1            2.65              2.65   
1   2010-11-04            Eating          4           10.44              2.61   
2   2010-11-04      Housekeeping          2           15.06              7.53   
3   2010-11-04  Meal_Preparation          9           67.29              7.48   
4   2010-11-04             Relax          5           64.01             12.80   
5   2010-11-04          Sleeping          2          474.33            237.17   
6   2010-11-04       Wash_Dishes          2           19.00              9.50   
7   2010-11-04              Work          2           25.13             12.57   
8   2010-11-05     Bed_to_Toilet          1            2.68              2.68   
9   2010-11-05            Eating          1            3.50              3.50   
10  2010-11-05      Housekeeping          1           90.01             90.01   
11  2010-11-05  Meal_Prepara

In [38]:
# =========================================
# Compare daily routine with baseline
# =========================================

routine_comparison = daily_routine.merge(
    baseline_table,
    on="activity",
    suffixes=("_daily", "_baseline")
)

print(routine_comparison.head(20))

          date          activity  frequency  total_duration  \
0   2010-11-04     Bed_to_Toilet          1            2.65   
1   2010-11-04            Eating          4           10.44   
2   2010-11-04      Housekeeping          2           15.06   
3   2010-11-04  Meal_Preparation          9           67.29   
4   2010-11-04             Relax          5           64.01   
5   2010-11-04          Sleeping          2          474.33   
6   2010-11-04       Wash_Dishes          2           19.00   
7   2010-11-04              Work          2           25.13   
8   2010-11-05     Bed_to_Toilet          1            2.68   
9   2010-11-05            Eating          1            3.50   
10  2010-11-05      Housekeeping          1           90.01   
11  2010-11-05  Meal_Preparation          3            8.02   
12  2010-11-05             Relax          2            9.76   
13  2010-11-05          Sleeping          3          800.32   
14  2010-11-05       Wash_Dishes          1            

In [35]:
# Frequency deviation
routine_comparison["frequency_deviation"] = (
    routine_comparison["frequency"]
    - routine_comparison["average_daily_frequency"]
)

# Duration deviation
routine_comparison["duration_deviation"] = (
    routine_comparison["average_duration_daily"]
    - routine_comparison["average_duration_baseline"]
)

# Start-time deviation
routine_comparison["start_time_deviation"] = (
    routine_comparison["average_start_hour_daily"]
    - routine_comparison["average_start_hour_baseline"]
)

# Round the results
routine_comparison[
    [
        "frequency_deviation",
        "duration_deviation",
        "start_time_deviation"
    ]
] = routine_comparison[
    [
        "frequency_deviation",
        "duration_deviation",
        "start_time_deviation"
    ]
].round(2)

print(routine_comparison.head(20))

          date          activity  frequency  total_duration  \
0   2010-11-04     Bed_to_Toilet          1            2.65   
1   2010-11-04            Eating          4           10.44   
2   2010-11-04      Housekeeping          2           15.06   
3   2010-11-04  Meal_Preparation          9           67.29   
4   2010-11-04             Relax          5           64.01   
5   2010-11-04          Sleeping          2          474.33   
6   2010-11-04       Wash_Dishes          2           19.00   
7   2010-11-04              Work          2           25.13   
8   2010-11-05     Bed_to_Toilet          1            2.68   
9   2010-11-05            Eating          1            3.50   
10  2010-11-05      Housekeeping          1           90.01   
11  2010-11-05  Meal_Preparation          3            8.02   
12  2010-11-05             Relax          2            9.76   
13  2010-11-05          Sleeping          3          800.32   
14  2010-11-05       Wash_Dishes          1            

KeyError: 'frequency_deviation'

In [40]:
routine_comparison = daily_routine.merge(
    baseline_table,
    on="activity",
    suffixes=("_daily", "_baseline")
)

routine_comparison["frequency_deviation"] = (
    routine_comparison["frequency"]
    - routine_comparison["average_daily_frequency"]
)

routine_comparison["duration_deviation"] = (
    routine_comparison["average_duration_daily"]
    - routine_comparison["average_duration_baseline"]
)

routine_comparison["start_time_deviation"] = (
    routine_comparison["average_start_hour_daily"]
    - routine_comparison["average_start_hour_baseline"]
)

In [41]:
routine_comparison.columns

Index(['date', 'activity', 'frequency', 'total_duration',
       'average_duration_daily', 'average_start_hour_daily',
       'average_daily_frequency', 'average_duration_baseline', 'min_duration',
       'max_duration', 'average_start_hour_baseline', 'frequency_deviation',
       'duration_deviation', 'start_time_deviation'],
      dtype='object')

In [42]:
variability = (
    daily_routine
    .groupby("activity")
    .agg(
        frequency_std=("frequency", "std"),
        duration_std=("average_duration", "std"),
        start_time_std=("average_start_hour", "std")
    )
    .reset_index()
)

variability

,activity,frequency_std,duration_std,start_time_std
0,Bed_to_Toilet,0.435390,0.985498,2.613255
1,Eating,1.356805,7.713036,3.528412
2,Housekeeping,0.390021,20.218954,2.987186
3,Meal_Preparation,3.554586,4.260761,2.147345
4,Relax,4.000634,11.306951,1.645044
5,Sleeping,0.883113,100.152102,6.712690
6,Wash_Dishes,0.483452,7.489216,4.328433
7,Work,1.236143,18.356466,3.579488


In [43]:
routine_comparison = routine_comparison.merge(
    variability,
    on="activity",
    how="left"
)

In [44]:
import numpy as np

routine_comparison["frequency_deviation_score"] = (
    routine_comparison["frequency_deviation"].abs()
    / routine_comparison["frequency_std"]
)

routine_comparison["duration_deviation_score"] = (
    routine_comparison["duration_deviation"].abs()
    / routine_comparison["duration_std"]
)

routine_comparison["start_time_deviation_score"] = (
    routine_comparison["start_time_deviation"].abs()
    / routine_comparison["start_time_std"]
)

In [45]:
score_columns = [
    "frequency_deviation_score",
    "duration_deviation_score",
    "start_time_deviation_score"
]

routine_comparison[score_columns] = (
    routine_comparison[score_columns]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
    .round(2)
)

In [46]:
routine_comparison[
    [
        "date",
        "activity",
        "frequency_deviation_score",
        "duration_deviation_score",
        "start_time_deviation_score"
    ]
].head(20)

,date,activity,frequency_deviation_score,duration_deviation_score,start_time_deviation_score
0,2010-11-04,Bed_to_Toilet,0.39,0.08,0.31
1,2010-11-04,Eating,1.58,0.98,0.05
2,2010-11-04,Housekeeping,2.10,0.63,0.86
3,2010-11-04,Meal_Preparation,0.48,0.08,0.04
4,2010-11-04,Relax,2.07,1.82,0.41
5,2010-11-04,Sleeping,0.03,0.04,0.94
6,2010-11-04,Wash_Dishes,1.55,0.30,0.33
7,2010-11-04,Work,0.13,0.25,0.85
8,2010-11-05,Bed_to_Toilet,0.39,0.05,0.07
9,2010-11-05,Eating,0.63,0.86,0.94
